# MammosighTR — YOLO11m-cls BI-RADS Sınıflandırma

Bu notebook, hazır `yolo_dataset.zip` veri setini kullanarak **YOLO11m-cls** modelini BI-RADS sınıflandırması için eğitir.

**Sınıflar:** `BIRADS0` | `BIRADS1_2` | `BIRADS4_5`

> Bu notebook'u çalıştırmadan önce sağ taraftaki **+ Add Input** butonuyla `mammosight_dataset_prep` notebook çıktısını ekle.

### ⚖️ Adil Karşılaştırma Notu
Tüm hiperparametreler (optimizer, lr, batch, dropout, augmentation) **YOLOv8m-cls** eğitimi ile **birebir aynıdır**.
İki model arasındaki tek fark **mimaridir** (C2f vs C3k2+C2PSA).

In [ ]:
# ── 1. Kurulum ──────────────────────────────────────────────────
!pip install ultralytics -q

import os, shutil, zipfile, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

from pathlib import Path
from ultralytics import YOLO
from sklearn.metrics import (
 classification_report, confusion_matrix,
 roc_curve, auc, RocCurveDisplay
)
from sklearn.preprocessing import label_binarize
import warnings
warnings.filterwarnings('ignore')

print(' Kurulum tamamlandı.')

In [ ]:
# ── 2. Veri Seti Hazırlama ───────────────────────────────────────
# Notebook output'undan zip dosyasını bul
possible_paths = [
 '/kaggle/input/mammosight-dataset-prep/yolo_dataset.zip',
 '/kaggle/working/yolo_dataset.zip',
]

# Glob ile de ara
found = glob.glob('/kaggle/input/**/yolo_dataset.zip', recursive=True)
if found:
 possible_paths = found + possible_paths

ZIP_PATH = None
for p in possible_paths:
 if os.path.exists(p):
  ZIP_PATH = p
  break

if ZIP_PATH is None:
 raise FileNotFoundError(
  "yolo_dataset.zip bulunamadı!\n"
  "Sağ taraftaki '+ Add Input' butonuyla "
  "'mammosight_dataset_prep' notebook çıktısını ekle."
 )

print(f'📦 Veri seti bulundu: {ZIP_PATH}')
print(f' Boyut: {os.path.getsize(ZIP_PATH)/1024/1024:.1f} MB')

# Çıkarma
DATASET_DIR = '/kaggle/working/dataset'
if os.path.exists(DATASET_DIR):
 shutil.rmtree(DATASET_DIR)
os.makedirs(DATASET_DIR, exist_ok=True)

print('📂 Zip çıkarılıyor...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
 zf.extractall(DATASET_DIR)

# Veri seti yapısını kontrol et
CLASS_NAMES = ['BIRADS0', 'BIRADS1_2', 'BIRADS4_5']
print('\n Veri Seti Dağılımı:')
total = 0
for split in ['train', 'val', 'test']:
 print(f' {split.upper()}:')
 for cls in CLASS_NAMES:
  path = os.path.join(DATASET_DIR, split, cls)
  count = len(glob.glob(os.path.join(path, '*.jpg'))) if os.path.exists(path) else 0
  total += count
  print(f' {cls}: {count} görsel')
print(f' Toplam: {total} görsel')

In [ ]:
# ── 3. YOLO11m-cls Eğitimi ───────────────────────────────────────
MODEL_NAME = 'yolo11m-cls'
EPOCHS  = 50
IMG_SIZE = 512
BATCH  = 16    # YOLOv8 ile aynı
WORKERS = 4
PROJECT = '/kaggle/working/runs'
RUN_NAME = 'mammosight_yolo11m'

print(f' YOLO11 Eğitimi Başlıyor...')
print(f' Model : {MODEL_NAME}')
print(f' Epochs : {EPOCHS}')
print(f' ImgSize : {IMG_SIZE}')
print(f' Batch : {BATCH}')

try:
 model = YOLO(f'{MODEL_NAME}.pt')
 print(f' {MODEL_NAME} yüklendi')
except Exception as e:
 print(f' {MODEL_NAME} yüklenemedi: {e}')
 print('🔄 Fallback: yolov8m-cls kullanılıyor')
 MODEL_NAME = 'yolov8m-cls'
 RUN_NAME = 'mammosight_yolo11_fallback_yolov8m'
 model = YOLO(f'{MODEL_NAME}.pt')

# ╔══════════════════════════════════════════════════════════════╗
# ║ YOLOv8 ile BİREBİR AYNI hiperparametreler     ║
# ║ (Adil karşılaştırma / Fair comparison)      ║
# ╚══════════════════════════════════════════════════════════════╝
results = model.train(
 data   = DATASET_DIR,
 epochs  = EPOCHS,
 imgsz   = IMG_SIZE,
 batch   = BATCH,
 workers  = WORKERS,
 project  = PROJECT,
 name   = RUN_NAME,
 optimizer  = 'AdamW',
 patience  = 30,
 lr0   = 1e-4,
 lrf   = 1e-2,
 cos_lr  = True,
 warmup_epochs = 3,
 dropout  = 0.3,
 augment  = True,
 device  = 0,
 exist_ok  = True,
 verbose  = True,
)

BEST_MODEL_PATH = os.path.join(PROJECT, RUN_NAME, 'weights', 'best.pt')
print(f'\n Eğitim tamamlandı. En iyi model: {BEST_MODEL_PATH}')

In [ ]:
# ── 4. Test Seti Değerlendirmesi ─────────────────────────────────
print(' Test seti değerlendiriliyor...')
best_model = YOLO(BEST_MODEL_PATH)

y_true, y_pred, y_scores = [], [], []

for cls_idx, cls_name in enumerate(CLASS_NAMES):
 test_path = os.path.join(DATASET_DIR, 'test', cls_name)
 if not os.path.exists(test_path):
  continue
 imgs = glob.glob(os.path.join(test_path, '*.jpg'))
 for img_path in imgs:
  pred = best_model.predict(img_path, verbose=False, imgsz=IMG_SIZE)[0]
  probs = pred.probs.data.cpu().numpy()
  predicted_cls = int(probs.argmax())
  y_true.append(cls_idx)
  y_pred.append(predicted_cls)
  y_scores.append(probs)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_scores = np.array(y_scores)

print('\n📋 Sınıflandırma Raporu:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

acc = (y_true == y_pred).mean() * 100
print(f'🎯 Test Accuracy: {acc:.2f}%')

In [ ]:
# ── 5. Confusion Matrix & ROC Eğrileri ──────────────────────────
DISPLAY_NAMES = ['BI-RADS 0', 'BI-RADS 1-2', 'BI-RADS 4-5']
COLORS = ['#6C63FF', '#FF6584', '#43C59E']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#1a1a2e')

# -- Confusion Matrix --
ax = axes[0]
ax.set_facecolor('#16213e')
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(DISPLAY_NAMES, fontsize=11, color='white', rotation=15)
ax.set_yticklabels(DISPLAY_NAMES, fontsize=11, color='white')
ax.set_xlabel('Tahmin Edilen', fontsize=12, color='white')
ax.set_ylabel('Gerçek', fontsize=12, color='white')
ax.set_title('Confusion Matrix (Normalize)', fontsize=14, color='white', pad=15)
ax.tick_params(colors='white')
for spine in ax.spines.values():
 spine.set_edgecolor('#444')
for i in range(3):
 for j in range(3):
  color = 'white' if cm_norm[i, j] < 0.5 else '#1a1a2e'
  ax.text(j, i, f'{cm_norm[i,j]:.2f}\n({cm[i,j]})',
    ha='center', va='center', fontsize=10,
    color=color, fontweight='bold')

# -- ROC Eğrileri --
ax2 = axes[1]
ax2.set_facecolor('#16213e')
y_bin = label_binarize(y_true, classes=[0, 1, 2])
for i, (cls_name, color) in enumerate(zip(DISPLAY_NAMES, COLORS)):
 fpr, tpr, _ = roc_curve(y_bin[:, i], y_scores[:, i])
 roc_auc = auc(fpr, tpr)
 ax2.plot(fpr, tpr, color=color, lw=2.5,
    label=f'{cls_name} (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], 'w--', lw=1, alpha=0.5)
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontsize=12, color='white')
ax2.set_ylabel('True Positive Rate', fontsize=12, color='white')
ax2.set_title('ROC Eğrileri (One-vs-Rest)', fontsize=14, color='white', pad=15)
ax2.tick_params(colors='white')
for spine in ax2.spines.values():
 spine.set_edgecolor('#444')
legend = ax2.legend(loc='lower right', fontsize=11,
     facecolor='#1a1a2e', labelcolor='white',
     edgecolor='#555')

plt.suptitle(
 f'YOLO11m-cls — MammosighTR BI-RADS Sınıflandırma | Test Accuracy: {acc:.2f}%',
 fontsize=15, color='white', y=1.02, fontweight='bold'
)
plt.tight_layout()
out_fig = '/kaggle/working/yolo11_results.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight',
   facecolor=fig.get_facecolor())
plt.show()
print(f' Grafik kaydedildi: {out_fig}')

In [ ]:
# ── 6. Eğitim Eğrileri ──────────────────────────────────────────
results_csv = os.path.join(PROJECT, RUN_NAME, 'results.csv')
if os.path.exists(results_csv):
 df_res = pd.read_csv(results_csv)
 df_res.columns = df_res.columns.str.strip()

 fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 fig.patch.set_facecolor('#1a1a2e')

 # Loss
 ax = axes[0]
 ax.set_facecolor('#16213e')
 loss_cols = [c for c in df_res.columns if 'loss' in c.lower()]
 for col in loss_cols:
  ax.plot(df_res['epoch'], df_res[col], label=col, lw=2)
 ax.set_xlabel('Epoch', color='white')
 ax.set_ylabel('Loss', color='white')
 ax.set_title('Eğitim / Doğrulama Loss', color='white', fontsize=13)
 ax.tick_params(colors='white')
 ax.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')
 for spine in ax.spines.values(): spine.set_edgecolor('#444')

 # Accuracy (sadece top1)
 ax2 = axes[1]
 ax2.set_facecolor('#16213e')
 acc_cols = [c for c in df_res.columns
    if ('acc' in c.lower() or 'top1' in c.lower()) and 'top5' not in c.lower()]
 for col in acc_cols:
  ax2.plot(df_res['epoch'], df_res[col], label=col, lw=2)
 ax2.set_xlabel('Epoch', color='white')
 ax2.set_ylabel('Accuracy', color='white')
 ax2.set_title('Top-1 Accuracy', color='white', fontsize=13)
 ax2.tick_params(colors='white')
 ax2.legend(facecolor='#1a1a2e', labelcolor='white', edgecolor='#555')
 for spine in ax2.spines.values(): spine.set_edgecolor('#444')

 plt.suptitle('YOLO11m-cls — Eğitim Eğrileri', color='white', fontsize=14, fontweight='bold')
 plt.tight_layout()
 curve_fig = '/kaggle/working/yolo11_training_curves.png'
 plt.savefig(curve_fig, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
 plt.show()
 print(f' Eğitim eğrileri kaydedildi: {curve_fig}')
else:
 print('results.csv bulunamadı, eğitim eğrileri atlandı.')

In [ ]:
# ── 7. Model Özeti & Çıktılar ────────────────────────────────────
from sklearn.metrics import roc_auc_score

print('=' * 50)
print(' YOLO11m-cls — SONUÇ ÖZETİ')
print('=' * 50)
print(f' Model : {MODEL_NAME}')
print(f' Epochs : {EPOCHS}')
print(f' Test Acc : {acc:.2f}%')
try:
 roc_macro = roc_auc_score(
  label_binarize(y_true, classes=[0,1,2]),
  y_scores, multi_class='ovr', average='macro'
 )
 print(f' ROC-AUC : {roc_macro:.4f} (macro-average)')
except:
 pass
print('=' * 50)
print(f' Best model: {BEST_MODEL_PATH}')
print(' Çıktı dosyaları:')
for f in glob.glob('/kaggle/working/*.png') + [BEST_MODEL_PATH]:
 if os.path.exists(f):
  print(f'  {f}')
print('=' * 50)

# Sonuçları Drive'a da kaydet
import shutil
for f in glob.glob('/kaggle/working/*.png'):
if os.path.exists(os.path.join(PROJECT, RUN_NAME, 'results.csv')):
